In [6]:

import wandb
import numpy as np
import joblib
from SRC.helper_functions.preprocessing import processor, prep_x_for_tf_horiz, prep_y_for_tf_horiz
import polars as pl

In [7]:
'''list of all sites    "sites": [
        "06936530", "06930000", "06900050", "06925250", "06929900",
        "06897500", "06901250", "06893970", "06928420", "06899700",
        "06935997", "06893620", "06906150", "06918440", "06932000",
        "06894000", "06935955", "06923250", "06900640", "06909500",
        "06901500", "06907700", "06893830", "06928380", "06896000",
        "06935770", "06919500", "06893390", "06920520", "06933500",
        "06928300", "06921600", "06900800", "06906300", "06897000",
        "06918740", "06909950", "06936475", "06908000", "06935850",
        "06902995", "06893820", "06930060", "06820500", "06899500",
        "06923940", "06935755", "06904500", "06910230", "06904650",
        "06927000", "06917060", "06921720", "06918460", "06921200",
        "06905500", "06918493", "06928000", "06899900", "06917630",
        "06901205", "06923950", "06894200", "06928330", "06906800",
        "06928359", "06893940", "06927240", "06893150", "06893557",
        "06917560", "06821080", "06921070", "06935830", "06935980",
        "06902000", "06896400", "06918060", "06926290", "06921590",
        "06893578", "06930015", "06928320", "06893750", "06895000",
        "06910750", "06934000", "06935890", "06821150", "06893500",
        "06906000", "06896900",
    ]
'''
config = {
    "input_cols": [
        "latitude",
        "longitude",
        "streamflow_cfs_mean",
        "gage_height_ft_mean",
        "precipitation_mm",
        "temperature_c",
        "specific_humidity_kgkg",
    ], 
    "target": "streamflow_cfs_mean",
    "train_split": 0.8,
    "val_split": 0.9,
    "start_date": "2020-01-01",
    "end_date": "2020-12-31",
    "file_path": "flood-dataset-missouri",
    "file_name": "flood_model_missouri",
    "table": "wandb.flood_model_missouri",
    "lag_window": 3,
    "sites": [
        "06936530", "06930000",
    ]
}

pcr = processor(config) 
pcr.pull_duckdb()
(
    train_X_scaled,
    val_X_scaled,
    test_X_scaled,
    train_y_scaled,
    val_y_scaled,
    test_y_scaled,
) = pcr.return_outputs()

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
import numpy as np


timesteps = config.get('lag_window', 3)
drop_col = ['latitude', 'longitude', 'site_id', 'observation_hour']

X_train = prep_x_for_tf_horiz(train_X_scaled, drop_col, timesteps) 
X_val = prep_x_for_tf_horiz(val_X_scaled, drop_col, timesteps)

y_train = prep_y_for_tf_horiz(train_y_scaled, train_X_scaled["site_id"], timesteps)
y_val = prep_y_for_tf_horiz(val_y_scaled, val_X_scaled["site_id"], timesteps)

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

model = Sequential([
    GRU(64, activation='tanh', return_sequences=True,),
    GRU(32, activation='tanh'),
    Dense(len(config['sites']), activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

X_train shape: (268, 3, 10)
y_train shape: (268, 2)


In [9]:

history = model.fit(X_train, y_train, epochs=2, batch_size=1, validation_data=(X_val, y_val))

Epoch 1/2
268/268 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.2971 - mae: 0.2315 - val_loss: 0.0817 - val_mae: 0.1208
Epoch 2/2
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1489 - mae: 0.1443 - val_loss: 0.0868 - val_mae: 0.1721


In [10]:
y_val

array([[-0.23173411, -0.49533084],
       [-0.14203388, -0.4955196 ],
       [ 0.20173348, -0.49493913],
       [ 3.45800211, -0.49565323],
       [ 0.90229727, -0.4957167 ],
       [ 0.34054333, -0.49573675],
       [ 0.12857016, -0.49575095],
       [ 0.01531736, -0.49574343],
       [-0.04982806, -0.49575345],
       [-0.09091978, -0.49574844],
       [-0.12399361, -0.49574343],
       [-0.15005178, -0.49574343],
       [-0.16809205, -0.49574343],
       [-0.18112113, -0.49575345],
       [-0.19039182, -0.49573341],
       [-0.19966252, -0.4954294 ],
       [-0.21118825, -0.49557054],
       [-0.22020838, -0.4953559 ],
       [-0.20818153, -0.49109806],
       [-0.18813679, -0.49497672],
       [ 0.69683865, -0.49553547],
       [ 0.25435093, -0.49563903],
       [ 0.08046278, -0.49566074],
       [-0.01474976, -0.49551291],
       [-0.07087504, -0.49548285],
       [-0.09893768, -0.49529159],
       [ 4.2472639 , -0.49483808],
       [ 2.53594389, -0.49554298],
       [ 1.16789013,